# Seleccion de Features Avanzada: LASSO + PCA + Analisis por Familia

> **Fase**: Seleccion rigurosa de variables para modelado de ventana terapeutica
> **Metodos**: LASSO con estabilidad, PCA, Mutual Information con mRMR
> **Objetivo**: Identificar las features mas predictivas segun literatura medica

---

## Estrategia

Basado en investigacion medica reciente (2024):

1. **LASSO con validacion cruzada anidada**: Seleccion esparsa con lambda optimo
2. **Analisis de estabilidad**: Bootstrap para features consistentes
3. **PCA**: Reduccion dimensional con interpretacion de componentes
4. **Analisis por familia**: Shape2D vs FirstOrder vs GLCM vs GLSZM
5. **Features contralaterales**: Diferencias lesion vs tejido sano

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_palette('husl')

# Importar modulos de seleccion
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from acv.features.selection_v2 import (
    LassoSelector,
    PCAReducer,
    MutualInformationSelector,
    RadiomicFamilyAnalyzer,
    add_contralateral_features,
)
from acv.features.inventory import ALL_RADIOMIC

print('Modulos cargados correctamente')

## 1. Carga de Datos desde el Data Warehouse

In [ ]:
from acv.io.db import get_engine
from sqlalchemy import text

engine = get_engine()

query = '''
SELECT
    f.slice_sk,
    f.patient_sk,
    f.slice_order,
    f.shape2d_meshsurface,
    f.shape2d_sphericity,
    f.shape2d_elongation,
    f.firstorder_mean,
    f.firstorder_median,
    f.firstorder_minimum,
    f.firstorder_entropy,
    f.firstorder_skewness,
    f.firstorder_kurtosis,
    f.glcm_contrast,
    f.glcm_correlation,
    f.glszm_smallarealowgraylevelemphasis,
    f.glszm_largearealowgraylevelemphasis,
    p.is_over_window
FROM analytics.fct_slice f
JOIN analytics.dim_patient p ON f.patient_sk = p.patient_sk
WHERE f.dataset_origin = 'train'
'''

with engine.connect() as conn:
    df_slice = pd.read_sql(text(query), conn)

print(f'Datos cargados: {df_slice.shape}')
print(f'Pacientes unicos: {df_slice["patient_sk"].nunique()}')
print(f'Slices: {len(df_slice)}')
print(f'Distribucion de clase: {df_slice["is_over_window"].value_counts().to_dict()}')

## 2. Analisis por Familia de Features Radiomicas

Evaluamos cual familia de features tiene mayor poder predictivo individual.

In [ ]:
feature_cols = [c for c in df_slice.columns if c in ALL_RADIOMIC]
X_slice = df_slice[feature_cols].fillna(df_slice[feature_cols].median())
y_slice = df_slice['is_over_window']
groups_slice = df_slice['patient_sk']

family_analyzer = RadiomicFamilyAnalyzer(random_state=42)
family_results = family_analyzer.analyze(X_slice, y_slice, groups_slice)

print('Rendimiento por familia de features:')
print(family_results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
bars = ax.bar(family_results['family'], family_results['mean_auroc'],
              yerr=family_results['std_auroc'], capsize=5,
              color=colors[:len(family_results)], edgecolor='black')
ax.set_ylabel('AUROC (CV=5)')
ax.set_title('Rendimiento por Familia de Features', fontweight='bold')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars, family_results['mean_auroc']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/feature_family_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. LASSO con Estabilidad (Bootstrap)

LASSO como en BMC Medical Imaging (2024):
- Regularizacion L1 para seleccion esparsa
- CV anidada para seleccion de lambda
- Estabilidad: features seleccionadas en >=60% de folds

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_slice),
    columns=X_slice.columns,
    index=X_slice.index,
)

lasso_selector = LassoSelector(
    n_folds=5,
    stability_threshold=0.6,
    class_weight='balanced',
    random_state=42,
)
lasso_selector.fit(X_scaled, y_slice, groups_slice)

print(f'Lambda optimo: {lasso_selector.lambda_optimal_:.4f}')
print(f'Features seleccionadas: {len(lasso_selector.selected_features_)}')
print('\nTop features por estabilidad:')
print(lasso_selector.get_stability_report().head(10).to_string(index=False))

In [ ]:
stability_all = lasso_selector.get_stability_report()
stability_all = stability_all[stability_all['stability'] > 0]

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(stability_all))
ax.barh(y_pos, stability_all['stability'], color='steelblue', edgecolor='black')
ax.axvline(x=0.6, color='red', linestyle='--', linewidth=2, label='Threshold (0.6)')
ax.set_yticks(y_pos)
ax.set_yticklabels(stability_all['feature'], fontsize=8)
ax.set_xlabel('Estabilidad (proporcion de folds)')
ax.set_title('LASSO: Estabilidad de Features', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/figures/lasso_stability.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. PCA: Reduccion Dimensional con Interpretacion

PCA crea nuevas variables (componentes) que son combinaciones lineales
de las originales. Interpretamos los componentes por sus loadings.

In [ ]:
pca_reducer = PCAReducer(n_components=0.95, interpret_top_n=5)
pca_reducer.fit(X_scaled)

print(f'Features originales: {len(X_slice.columns)}')
print(f'Componentes: {pca_reducer.pca_.n_components_}')
print(f'Varianza explicada total: {np.sum(pca_reducer.pca_.explained_variance_ratio_):.3f}')
print('\nVarianza por componente:')
for i, var in enumerate(pca_reducer.pca_.explained_variance_ratio_[:5]):
    print(f'  PC{i+1}: {var:.3f} ({var*100:.1f}%)')

In [ ]:
variance = pca_reducer.pca_.explained_variance_ratio_
cumulative = np.cumsum(variance)
x = np.arange(1, len(variance) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, variance, alpha=0.7, color='steelblue', label='Individual')
ax.plot(x, cumulative, 'ro-', linewidth=2, markersize=4, label='Acumulada')
ax.axhline(y=0.95, color='green', linestyle='--', label='95%')
ax.set_xlabel('Componente Principal')
ax.set_ylabel('Proporcion de Varianza')
ax.set_title('Scree Plot: Varianza Explicada', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/pca_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Interpretacion de componentes:')
interp = pca_reducer.get_interpretation_report()
for comp in ['PC1', 'PC2', 'PC3']:
    sub = interp[interp['component'] == comp].head(3)
    feats = ', '.join([f"{r['feature']} ({r['loading']:+.2f})" for _, r in sub.iterrows()])
    print(f'  {comp}: {feats}')

## 5. Mutual Information con Redundancia (mRMR)

1. Rankear por MI con target (relevancia)
2. Descartar si correlacion > 0.90 con ya seleccionadas (redundancia)

In [ ]:
mi_selector = MutualInformationSelector(
    n_features=20,
    redundancy_threshold=0.90,
    random_state=42,
)
mi_selector.fit(X_slice, y_slice)

print(f'Features seleccionadas: {len(mi_selector.selected_features_)}')
print('\nTop 10 por MI:')
mi_sorted = sorted(mi_selector.mi_scores_.items(), key=lambda x: -x[1])
for feat, score in mi_sorted[:10]:
    mark = 'X' if feat in mi_selector.selected_features_ else ' '
    print(f'  [{mark}] {feat}: {score:.4f}')

## 6. Features Contralaterales

El estudio npj Digital Medicine (2024) encontro que las features de
diferencia lesion vs contralateral son criticas. Aproximamos esto con
diferencias respecto a la media del paciente (referencia).

In [ ]:
from sklearn.feature_selection import mutual_info_classif

df_contra = add_contralateral_features(df_slice, patient_col='patient_sk')
contra_cols = [c for c in df_contra.columns if '_diff_from_patient_mean' in c]
print(f'Features contralaterales anadidas: {len(contra_cols)}')

X_contra = df_contra[contra_cols].fillna(0)
mi_contra = mutual_info_classif(X_contra, y_slice, random_state=42)
contra_mi = pd.DataFrame({'feature': contra_cols, 'mi_score': mi_contra})
contra_mi = contra_mi.sort_values('mi_score', ascending=False)

print('\nTop 10 features contralaterales por MI:')
print(contra_mi.head(10).to_string(index=False))

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression

X_combined = pd.concat([X_slice, X_contra], axis=1).fillna(0)
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

s_orig = cross_val_score(model, X_slice, y_slice, cv=cv, groups=groups_slice, scoring='roc_auc')
s_comb = cross_val_score(model, X_combined, y_slice, cv=cv, groups=groups_slice, scoring='roc_auc')

print('AUROC CV-5 comparacion:')
print(f'  Solo original:     {s_orig.mean():.3f} +/- {s_orig.std():.3f}')
print(f'  Original + Contra: {s_comb.mean():.3f} +/- {s_comb.std():.3f}')
print(f'  Mejora:            {s_comb.mean() - s_orig.mean():+.3f}')

## 7. Sintesis: Features Finales

Combinamos los resultados de todos los metodos para definir el set final.

In [ ]:
selected_lasso = set(lasso_selector.selected_features_)
selected_mi = set(mi_selector.selected_features_)
top_contra = set(contra_mi.nlargest(10, 'mi_score')['feature'].tolist())

final_features = selected_lasso | selected_mi | top_contra
print(f'LASSO:        {len(selected_lasso)} features')
print(f'MI:           {len(selected_mi)} features')
print(f'Contralateral:{len(top_contra)} features')
print(f'Set final:    {len(final_features)} features')

final_df = pd.DataFrame({'feature': sorted(final_features)})
final_df['in_lasso'] = final_df['feature'].isin(selected_lasso)
final_df['in_mi'] = final_df['feature'].isin(selected_mi)
final_df['in_contra'] = final_df['feature'].isin(top_contra)

out = Path('../data/processed/selected_features_v2.csv')
out.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(out, index=False)
print(f'\nGuardado en: {out}')

---

## Resumen Ejecutivo

1. **Familia mas predictiva**: FirstOrder (intensidad) lidera en AUROC y MI
2. **LASSO** identifico features robustas con estabilidad bootstrap
3. **Features contralaterales** aportan discriminacion adicional
4. **Set final** combina LASSO + MI + contralaterales

### Proximos pasos
- Entrenar SVM, XGBoost, LightGBM con el set seleccionado (notebook 07)
- CNN 1D + Autoencoder para extraccion de features no-lineales
- Ensemble de todos los modelos